# Studi Kasus 1 — Audit Kualitas Data dan Analitik Penjualan
**Materi:** Make Sense of Data with Analysis and AI  
**Jenis:** analisis data eksploratif dengan dataset sintetis

Notebook ini memperlihatkan mengapa kualitas data perlu diperiksa sebelum analisis dan keputusan.

## 1. Rumusan masalah dan batasan
Dataset transaksi fiktif sengaja berisi nilai kosong, format tanggal berbeda, duplikasi, dan harga tidak valid. Tujuan analisis adalah membangun KPI yang transparan. Data tidak mewakili perusahaan nyata; pencilan ditandai untuk tinjauan, bukan otomatis dianggap salah.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(41)
n = 120
dates = pd.date_range("2025-01-01", periods=n, freq="3D")
products = ["Kopi", "Teh", "Cokelat", "Susu"]
df_raw = pd.DataFrame({
    "order_id": [f"ORD-{i:04d}" for i in range(n)],
    "tanggal": dates.strftime("%Y-%m-%d"),
    "produk": rng.choice(products, n),
    "kuantitas": rng.integers(1, 12, n),
    "harga_satuan": rng.choice([12000, 15000, 18000, 22000, 25000], n),
})

df_raw.loc[[4, 22, 63], "kuantitas"] = np.nan
df_raw.loc[[9, 40], "tanggal"] = ["31/02/2025", "2025/13/40"]
df_raw.loc[[17, 81], "harga_satuan"] = [-5000, 0]
df_raw = pd.concat([df_raw, df_raw.iloc[[10, 25]]], ignore_index=True)
df_raw.head()

In [ ]:
print("Ukuran data:", df_raw.shape)
print("\nTipe data:")
display(df_raw.dtypes.to_frame("dtype"))
print("\nNilai kosong:")
display(df_raw.isna().sum().to_frame("jumlah_kosong"))
print("\nDuplikasi baris:", int(df_raw.duplicated().sum()))

## 2. Pembersihan data
Aturan yang digunakan: parsing tanggal dengan `errors='coerce'`, menghapus duplikat persis, mengubah kolom numerik, dan mengecualikan transaksi dengan tanggal/kuantitas/harga yang tidak valid dari analisis KPI. Baris yang dikecualikan tetap dicatat untuk audit.

In [ ]:
df = df_raw.copy()
df["tanggal"] = pd.to_datetime(df["tanggal"], errors="coerce", dayfirst=True)
df["kuantitas"] = pd.to_numeric(df["kuantitas"], errors="coerce")
df["harga_satuan"] = pd.to_numeric(df["harga_satuan"], errors="coerce")

before = len(df)
df = df.drop_duplicates()
after_dedup = len(df)
valid_mask = (
    df["tanggal"].notna()
    & df["kuantitas"].notna()
    & (df["kuantitas"] > 0)
    & df["harga_satuan"].notna()
    & (df["harga_satuan"] > 0)
)
df_excluded = df.loc[~valid_mask].copy()
df_clean = df.loc[valid_mask].copy()
df_clean["pendapatan"] = df_clean["kuantitas"] * df_clean["harga_satuan"]

print(f"Baris awal: {before}")
print(f"Baris setelah hapus duplikat: {after_dedup}")
print(f"Baris dikecualikan dari KPI: {len(df_excluded)}")
print(f"Baris bersih: {len(df_clean)}")
display(df_excluded.head(10))

In [ ]:
kpi = pd.Series({
    "Jumlah transaksi bersih": len(df_clean),
    "Total unit": int(df_clean["kuantitas"].sum()),
    "Total pendapatan": float(df_clean["pendapatan"].sum()),
    "Rata-rata pendapatan/transaksi": float(df_clean["pendapatan"].mean()),
    "Jumlah produk unik": int(df_clean["produk"].nunique())
}, name="Nilai")
display(kpi.to_frame())

In [ ]:
monthly = (df_clean.assign(bulan=df_clean["tanggal"].dt.to_period("M").astype(str))
           .groupby("bulan", as_index=False)["pendapatan"].sum())
by_product = (df_clean.groupby("produk", as_index=False)["pendapatan"]
              .sum().sort_values("pendapatan", ascending=False))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(monthly["bulan"], monthly["pendapatan"], marker="o")
axes[0].set_title("Pendapatan per bulan")
axes[0].set_ylabel("Pendapatan (Rp)")
axes[0].tick_params(axis="x", rotation=45)
axes[1].bar(by_product["produk"], by_product["pendapatan"])
axes[1].set_title("Pendapatan per produk")
axes[1].tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()
display(by_product)

In [ ]:
q1, q3 = df_clean["pendapatan"].quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers = df_clean.loc[
    (df_clean["pendapatan"] < low) | (df_clean["pendapatan"] > high),
    ["order_id", "tanggal", "produk", "kuantitas", "harga_satuan", "pendapatan"]
]
print(f"Batas IQR: {low:,.0f} sampai {high:,.0f}")
print("Pencilan yang perlu ditinjau:", len(outliers))
display(outliers)

## 3. Interpretasi dan kesimpulan
- KPI dihitung setelah aturan pembersihan yang dinyatakan eksplisit.
- Pencilan adalah sinyal untuk pemeriksaan, bukan bukti kesalahan.
- Data sintetis ini hanya menunjukkan alur kerja; keputusan nyata membutuhkan definisi bisnis, audit sumber, serta konteks diskon, retur, pajak, dan biaya.
- Sebelum memakai AI, pastikan target, fitur, kualitas label, dan dampak keputusan telah ditentukan.